# Notebook 11 — Collections, itertools & functools

| | |
|---|---|
| **Track** | AI Engineering · `03-python-foundations/exercises/` |
| **Level** | Intermediate |
| **Pattern** | *Concept* → *Runnable demo* → *Your turn* → *Solution* |

**Prerequisites:** `10-testing-debugging.ipynb`

**Next up:** `12-logging-for-pipelines.ipynb`

---

## Learning objectives

- Choose `Counter`, `defaultdict`, and `deque` for ingestion-shaped code.
- Compose iterators with `itertools` instead of nested loops.
- Memoize pure lookups with `functools.lru_cache` where appropriate.

## Table of contents

1. **`collections` recipes**
2. **`itertools` building blocks**
3. **`functools` helpers**
4. **Progressive drills — grouping → chaining iterators → cached stubs**
5. **Exercise — overlapping windows generator**

---

## How to use this notebook

1. Run cells **top to bottom** the first time through.
2. Re-run and **change values** to test your understanding.
3. Do **Your turn** cells before opening solutions (HTML `<details>` blocks or later cells).

---


## 1 · `collections` recipes

*Explanation:* **`defaultdict`** drops boilerplate when bucketing retrieval hits by `doc_id`. **`Counter`** answers “top noisy tokens” questions without manual dict increments.


In [ ]:
from collections import Counter, defaultdict

hits = [
    {"doc_id": "wiki:42", "score": 0.9},
    {"doc_id": "ticket:7", "score": 0.4},
    {"doc_id": "wiki:42", "score": 0.8},
]
by_doc: dict[str, list[float]] = defaultdict(list)
for h in hits:
    by_doc[h["doc_id"]].append(h["score"])

tokens = "chunk chunk tokens tokens tokens".split()
print(dict(by_doc))
print(Counter(tokens).most_common(2))


## 2 · `itertools` building blocks

*Explanation:* **`chain`** flattens batches; **`islice`** trims infinite iterators — patterns that mirror streaming corpora.


In [ ]:
from itertools import chain, islice

batch_a = ["doc-a1", "doc-a2"]
batch_b = ["doc-b1"]
for doc_id in chain(batch_a, batch_b):
    print("ingest", doc_id)


def backoff_seconds():
    """Yield suggested retry delays without storing the whole schedule."""
    base = 0.05
    while True:
        yield base
        base = min(base * 2, 2.0)


print(list(islice(backoff_seconds(), 5)))


## 3 · `functools` helpers

*Explanation:* **`lru_cache`** caches immutable-key lookups — useful for stub embedding dictionaries during offline tests.


In [ ]:
from functools import lru_cache


@lru_cache(maxsize=256)
def fake_embed_dimension(text: str) -> int:
    # Pretend expensive tokenizer lookup
    return len(text.split()) * 4


print(fake_embed_dimension("rag pipeline"))
print(fake_embed_dimension("rag pipeline"))  # hits cache


---

## Progressive drills — **easy → harder**

Bridge **tabular ingestion** utilities with lazy iterators — memory stays flat while sophistication rises.


### A · Easiest — bucket lines by prefix

Route stderr vs stdout-shaped records without nested `if` chains.


In [ ]:
from collections import defaultdict

rows = ["OUT: ok", "ERR: timeout", "OUT: shipped"]
bucket = defaultdict(list)
for row in rows:
    kind, _, msg = row.partition(": ")
    bucket[kind].append(msg)

print(dict(bucket))


### B · Medium — flatten nested IDs

Mirror merging shards before dedupe.


In [ ]:
from itertools import chain

nested = [["u1", "u2"], ["u3"], ["u4", "u5"]]
flat = list(chain.from_iterable(nested))
print(flat)


### C · Harder — bounded batches without storing everything

Chunk embedding batches from any iterable.


In [ ]:
from collections.abc import Iterator


def batched_lines(lines: Iterator[str], size: int):
    batch: list[str] = []
    for line in lines:
        batch.append(line)
        if len(batch) >= size:
            yield batch
            batch = []
    if batch:
        yield batch


demo = batched_lines(iter(["a", "b", "c", "d", "e"]), 2)
print(list(demo))


### Exercise — overlapping windows

Implement `windowed(text: str, k: int)` as a **generator** yielding each substring of length `k` with stride `1`. Raise `ValueError` if `k < 1` or `k > len(text)`.

Example: `list(windowed("abcd", 2)) == ["ab", "bc", "cd"]`.


In [ ]:
from collections.abc import Iterator


def windowed(text: str, k: int) -> Iterator[str]:
    raise NotImplementedError


assert list(windowed("abcd", 2)) == ["ab", "bc", "cd"]
assert list(windowed("hi", 2)) == ["hi"]
print("OK")


### Solution — windowed

<details>
<summary>Click to expand</summary>

```python
from collections.abc import Iterator

def windowed(text: str, k: int) -> Iterator[str]:
    if k < 1 or k > len(text):
        raise ValueError("bad window")
    for i in range(len(text) - k + 1):
        yield text[i : i + k]
```

</details>
